In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold 파이프라인 실행: 전체 Target(적재 순서) × 전체 소스
# MAGIC
# MAGIC `dq_run`이 DQ 검사와 정제를 한 노트북에서 끝내는 것과 같은 방식입니다. 이 노트북은 Target 하나마다
# MAGIC **소스 목록 확정 → 소스별 Mapping(물리 저장) → Target Validation → Gold/격리/MIGRATION_TRACE 저장**을 **끝까지 마친 뒤**
# MAGIC 다음 Target으로 넘어갑니다. Target도 소스도 하드코딩하지 않습니다 — 둘 다 `meta.mapping_definition`에서
# MAGIC 승인된(FINAL_MIGRATION_APPLY_YN=Y, REVIEW_STATUS 승인) 것만 매번 다시 찾습니다.
# MAGIC
# MAGIC **실행 순서는 알파벳순이 아니라 `TARGET_LOAD_ORDER`** 기준입니다(예: CUSTOMER 20 → CONTRACT 30 → COUNSEL 40).
# MAGIC CONTRACT가 CUSTOMER를 FK로 참조하는 등 Target 간 참조 관계가 있어서, 부모가 먼저 만들어져야 합니다.
# MAGIC
# MAGIC 실행 가능한 Target은 `mapping_config.SUPPORTED_TARGET_TABLES`와의 교집합으로만 정해집니다(지금은 COUNSEL뿐).
# MAGIC 매핑 정의에 CUSTOMER·CONTRACT로 승인된 매핑이 있어도, 이 엔진은 1:1 변환만 지원해서 자동으로 제외됩니다
# MAGIC (중복 제거·개체 통합이 필요한 Target에 그대로 실행하면 레코드가 오염됩니다).
# MAGIC
# MAGIC **한 Target에서 오류가 나도 다음 Target 처리에는 영향이 없습니다** (예외를 잡고 FAILED로 기록한 뒤 계속 진행).
# MAGIC
# MAGIC 후보(`gold_candidate.<target>`)는 물리 테이블이라 노트북이 분리돼 있어도 이어집니다. 소스나 Target 하나만
# MAGIC 깊게 디버깅할 때는 `mapping_run.py`(단일 소스)나 `gold_validation_run.py`(재검증 전용)를 따로 쓰세요.

# COMMAND ----------

SAVE_GOLD = True       # False면 Validation 결과만 확인하고 Gold/격리/MIGRATION_TRACE에는 저장하지 않음

# COMMAND ----------

import sys
import traceback
from pyspark.sql import functions as F

if "PROJECT_ROOT" in dir() and PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
try:
    import src.mapping.mapping_config as mcfg
    import src.mapping.mapping_engine as engine_mod
    import src.gold.gold_validation_config as gcfg
    import src.gold.gold_validation_runner as gv
except ModuleNotFoundError:
    import mapping_config as mcfg
    import mapping_engine as engine_mod
    import gold_validation_config as gcfg
    import gold_validation_runner as gv


def _show(df, n=10):
    try:
        display(df.limit(n))
    except NameError:
        df.show(n, False)


# COMMAND ----------

# MAGIC %md ## 1. Target 목록 확정 (적재 순서)

# COMMAND ----------

engine = engine_mod.MappingEngine.from_tables(spark)
target_tables = engine.discover_target_tables()
print(f"실행 대상 Target (적재 순서): {target_tables}")
if not target_tables:
    raise RuntimeError("실행 가능한 Target이 없습니다 (meta.mapping_definition에 승인된 매핑이 없거나, "
                       "전부 mapping_config.SUPPORTED_TARGET_TABLES 밖입니다).")

# COMMAND ----------

# MAGIC %md ## 2. Target별 파이프라인 (Mapping → Validation → 저장)
# MAGIC 한 Target의 매핑·검증·저장이 전부 끝나야 다음 Target으로 넘어갑니다.

# COMMAND ----------

def run_pipeline_for_target(target_table: str) -> dict:
    print(f"\n{'=' * 70}\n🎯 Target: {target_table}\n{'=' * 70}")
    out = {"target_table": target_table}

    # ---- 2-1. 소스별 Mapping (물리 저장) ----
    sources = engine.discover_sources(target_table)
    print(f"승인된 소스: {sources}")
    if not sources:
        print("⚠️  승인된 소스가 없어 건너뜁니다.")
        out.update(status="SKIPPED (승인된 소스 없음)")
        return out

    mapping_rows = []
    for source in sources:
        silver_table = mcfg.silver_input_table(mcfg.SOURCE_SYSTEMS[source]["silver"])
        if not spark.catalog.tableExists(silver_table):
            print(f"  ⏭️  {source}: {silver_table} 없음 (건너뜀 - DQ가 아직 이 소스를 처리하지 않은 것으로 보임)")
            mapping_rows.append({"source": source, "status": "SKIPPED (Silver 없음)"})
            continue
        try:
            candidate, summary = engine.run(source, target_table)
        except NotImplementedError as e:
            print(f"  ⏭️  {source}: {e}")
            mapping_rows.append({"source": source, "status": "SKIPPED (미지원 Target)"})
            continue
        except ValueError as e:
            # 매핑 정의 자체의 오류(Source 컬럼 없음, 타입 불일치 등)는 이 소스만 건너뛰고 나머지 소스는 계속 진행한다.
            # 한 소스의 정의 문제가 같은 Target의 다른 소스까지 막으면 안 된다 (Silver 없음/미지원 Target과 같은 원칙).
            print(f"  ❌ {source}: 매핑 정의 오류로 건너뜀\n     {e}")
            mapping_rows.append({"source": source, "status": f"FAILED (정의 오류): {e}"})
            continue
        table = engine.save(candidate, summary)
        print(f"  ✅ {source}: {table}  (입력 {summary['input_count']} → 출력 {summary['output_count']}, "
             f"변환실패 {summary['conversion_error_rows']}, 미매핑 {summary['unmapped_code_rows']})")
        mapping_rows.append({
            "source": source, "status": "OK", "batch": summary["source_batch_id"],
            "input": summary["input_count"], "output": summary["output_count"],
            "conversion_errors": summary["conversion_error_rows"], "unmapped_codes": summary["unmapped_code_rows"],
        })
    _show(spark.createDataFrame(mapping_rows, samplingRatio=1.0), 10)

    ok_sources = [r["source"] for r in mapping_rows if r["status"] == "OK"]
    skipped_sources = [r["source"] for r in mapping_rows if r["status"] != "OK"]
    out["mapping_sources_ok"] = ok_sources
    out["mapping_sources_skipped"] = skipped_sources
    if not ok_sources:
        print("⚠️  실행된 소스가 하나도 없어 이 Target을 건너뜁니다 (Validation도 실행하지 않음).")
        out.update(status="SKIPPED (실행된 소스 없음)")
        return out
    if skipped_sources:
        print(f"⚠️  건너뛴 소스: {skipped_sources}. gold_candidate.{target_table.lower()}에는 이전에 저장된 값이 남아있을 수 있습니다.")

    # ---- 2-2. Target Validation ----
    validator = gv.TargetValidator.from_tables(spark)
    passed, failed, vsummary = validator.run(target_table)
    match = vsummary["input_count"] == vsummary["loaded_count"] + vsummary["quarantined_count"]
    print(f"\nvalidation_run_id: {vsummary['validation_run_id']}")
    print(f"입력 {vsummary['input_count']} = 적재 {vsummary['loaded_count']} + 격리 {vsummary['quarantined_count']}  "
         f"{'✅ 대사 일치' if match else '❌ 불일치'}")
    if vsummary["pk_dedup_fixed"]:
        print(f"PK 중복 보정: {vsummary['pk_dedup_fixed']}건")
    if vsummary["violations_by_rule"]:
        print("위반 규칙 분포:")
        for rule_id, cnt in sorted(vsummary["violations_by_rule"].items(), key=lambda kv: -kv[1]):
            print(f"  {rule_id:<28}{cnt}건")
    out.update(validation_run_id=vsummary["validation_run_id"], loaded_count=vsummary["loaded_count"],
              quarantined_count=vsummary["quarantined_count"], reconciled=match)

    # ---- 2-3. 저장 ----
    if SAVE_GOLD:
        tables = validator.save(passed, failed, vsummary)
        for label, tbl in tables.items():
            print(f"✅ {label}: {tbl}")
        out["tables"] = tables
    else:
        print("SAVE_GOLD=False: 저장하지 않았습니다.")

    out["status"] = "OK"
    return out


pipeline_results = []
for target_table in target_tables:
    try:
        result = run_pipeline_for_target(target_table)
    except Exception as e:
        print(f"❌ {target_table} 처리 중 오류: {e}")
        traceback.print_exc()
        result = {"target_table": target_table, "status": f"FAILED ({type(e).__name__})"}
    pipeline_results.append(result)

# COMMAND ----------

# MAGIC %md ## 3. 전체 요약

# COMMAND ----------

print("\n✨ Target별 실행 결과")
for r in pipeline_results:
    print(f"  {r['target_table']:<12}{r['status']}")

failed_targets = [r["target_table"] for r in pipeline_results if r["status"].startswith("FAILED")]
if failed_targets:
    print(f"\n⚠️  실패한 Target: {failed_targets}. 위 로그(트레이스백)를 확인하세요. "
         "다른 Target의 처리에는 영향을 주지 않았습니다.")

_show(spark.createDataFrame(
    [{"target_table": r["target_table"], "status": r["status"],
      "loaded": r.get("loaded_count"), "quarantined": r.get("quarantined_count")} for r in pipeline_results],
    "target_table string, status string, loaded long, quarantined long",   # 실패한 Target은 loaded/quarantined가 전부
), 10)                                                                     # None이라 타입 추론이 안 되므로 스키마를 명시한다

In [0]:
%sql
SELECT o.CALL_ST_DTM, o.CALL_END_DTM
FROM maps_databricks.silver_candidate.outbound o
LIMIT 5;

In [0]:
%sql
SELECT 
  SUM(CASE WHEN CALL_ST_DTM RLIKE '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$' THEN 1 ELSE 0 END) AS 초있음,
  SUM(CASE WHEN CALL_ST_DTM RLIKE '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}$' THEN 1 ELSE 0 END) AS `초없음`,
  COUNT(*) AS 전체
FROM maps_databricks.silver_candidate.outbound;
